# GraspMamba setup - Colab / Kaggle GPU

Enable GPU and Internet first. Upload/clone the repository and selected data. See `BASELINE_RESTORE.md`. CUDA compatibility must be checked in your actual runtime; this notebook has not been executed on hosted GPU here.


In [ ]:
from pathlib import Path
import os
import sys
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ.setdefault('MAX_JOBS', '2')
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/Grasp_Mamba_TS'), Path('/kaggle/working/Grasp_Mamba_TS')]
REPO = next((p for p in candidates if (p / 'train_network.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Upload/clone repo, then set REPO to its Linux folder and rerun this cell')
os.chdir(REPO)
# The original script calls python3 directly; put this kernel environment first.
os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ['PATH']
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache' if Path('/kaggle').exists() else '/content/hf-cache')
print('Repo:', REPO, '\nHF_HOME:', os.environ['HF_HOME'])


In [ ]:
import sys
import torch
import subprocess
print(sys.version)
print('torch', torch.__version__, 'CUDA wheel', torch.version.cuda, 'GPU enabled', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)
subprocess.run('nvcc --version', shell=True, check=False)


## Install packages
Keep the matched runtime torch/torchvision initially. Restart the kernel/session after this cell, then rerun the first cell. The original baseline imports CLIP, ruamel.yaml and seaborn; install these without editing the baseline source.


In [ ]:
%pip install -r requirements-notebook.txt

%pip install "git+https://github.com/openai/CLIP.git" ruamel.yaml seaborn


## Install CUDA extensions
Run after package installation/restart, with CUDA-enabled torch already present. A matching wheel may be used; otherwise nvcc/toolkit is required for compilation. Restart again if previously imported modules were replaced.


In [ ]:
%pip install --no-build-isolation --no-deps -r requirements-cuda.txt


In [ ]:
!python tools/check_notebook_environment.py


## Cache pretrained models
About 1.73 GB of safetensors plus small files. Uses HF_HOME; excludes redundant weight copies.


In [ ]:
!python tools/download_pretrained.py


In [ ]:
!python3 tools/check_notebook_environment.py


## Baseline: original training behavior with path options and tqdm
run_training.v1.sh keeps its original default configuration and now forwards explicit CLI flags. The cell below selects the original ++ loader and overrides only dataset/model locations. Change paths for mounted Kaggle inputs. Core train/validation, MSE, augmentation, optimizer steps, warmup and model modes stay original. The progress total shows actual original optimizer updates (batches_per_epoch - 1), without fixing the upstream off-by-one behavior.
Select TEXT_SOURCE='object' for the original object query, or 'instruction' for grasp_instructions. This selects one text source, not concatenation; training and validation use the same source. Keep the same choice when evaluating.


In [ ]:
DATASET_ROOT = REPO / 'data/grasp-anything'
ANNOTATION_ROOT = REPO / 'data/grasp-anything-pp'
TEXT_SOURCE = 'object'  # 'object': original query; 'instruction': detailed grasp instruction
VISION_MODEL = 'nvidia/MambaVision-L-1K'  # Or a complete local MambaVision-L snapshot
TEXT_MODEL = 'google/siglip-base-patch16-224'  # Or a complete local SigLIP snapshot

!MPLBACKEND=Agg bash run_training.v1.sh \
    --dataset grasp-anywhere \
    --text-source {TEXT_SOURCE} \
    --dataset-path "{DATASET_ROOT}" \
    --add-file-path "{ANNOTATION_ROOT}" \
    --vision-model "{VISION_MODEL}" \
    --text-model "{TEXT_MODEL}" \
    --description baseline_{TEXT_SOURCE}
